# GTFS 시계열 로더

폴더 구조 가정:
```
GTFS_CT/
├── 202103/   ← YYYYMM
│   ├── agency.txt
│   ├── stops.txt
│   ├── routes.txt
│   ├── trips.txt
│   ├── stop_times.txt
│   └── calendar.txt
├── 202203/
├── 202303/
└── 202403/   ← transfers.txt 추가
```

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
GTFS_DIR = Path(r"C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT")

# KTDB에서 구축한 파일 목록 (설명서 기준)
GTFS_FILES = ["agency", "stops", "routes", "trips", "stop_times", "calendar", "transfers"]

In [3]:
def load_gtfs_file(path: Path, year_month: str) -> pd.DataFrame | None:
    """단일 GTFS txt 로드 → year_month 컬럼 추가"""
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
        df["year_month"] = year_month          # ex) '202103'
        df["year"]       = int(year_month[:4]) # ex) 2021
        df["month"]      = int(year_month[4:]) # ex) 3
        return df
    except Exception as e:
        print(f"  [ERR] {path.name} ({year_month}): {e}")
        return None


def load_all_gtfs(gtfs_dir: Path) -> dict[str, pd.DataFrame]:
    """
    모든 연월 폴더를 순회하여 파일 종류별로 합산
    반환: {'stops': DataFrame, 'routes': DataFrame, ...}
    """
    result: dict[str, list[pd.DataFrame]] = {name: [] for name in GTFS_FILES}

    year_dirs = sorted([d for d in gtfs_dir.iterdir() if d.is_dir()])
    print(f"발견된 연월 폴더: {[d.name for d in year_dirs]}\n")

    for yd in year_dirs:
        ym = yd.name  # '202103' 등
        for name in GTFS_FILES:
            df = load_gtfs_file(yd / f"{name}.txt", ym)
            if df is not None:
                result[name].append(df)
                print(f"  [OK] {ym}/{name}.txt  {df.shape}")

    # 리스트 → 단일 DataFrame
    return {
        name: pd.concat(frames, ignore_index=True)
        for name, frames in result.items()
        if frames
    }

In [4]:
gtfs = load_all_gtfs(GTFS_DIR)

발견된 연월 폴더: ['대중교통GTFS', '도로망', '철도', '철도망', '행정경계']



In [5]:
# 로드 결과 요약
print("=== 테이블별 행 수 ===")
for name, df in gtfs.items():
    print(f"  {name:15s}: {len(df):>10,}행  |  연월: {sorted(df['year_month'].unique())}")

=== 테이블별 행 수 ===


---
## 시계열 분석 예시

In [6]:
# 연도별 정류장 수 추이
stops_ts = gtfs["stops"].groupby("year_month").size().reset_index(name="stop_count")
print("=== 연도별 정류장 수 ===")
print(stops_ts.to_string(index=False))

KeyError: 'stops'

In [ ]:
# 연도별 노선 수 × 교통수단 유형
# route_type 코드 (KTDB 기준)
ROUTE_TYPE = {
    0: "시내/농어촌/마을버스",
    1: "도시철도/경전철",
    2: "해운",
    3: "시외버스",
    4: "일반철도",
    5: "공항버스",
    6: "고속철도",
    7: "항공",
}

routes = gtfs["routes"].copy()
routes["transport_type"] = routes["route_type"].map(ROUTE_TYPE)

routes_ts = (
    routes.groupby(["year_month", "transport_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
print("=== 연도별 노선 수 × 교통수단 ===")
print(routes_ts.to_string(index=False))

In [ ]:
# 연도별 운행횟수(trips) 추이
trips_ts = gtfs["trips"].groupby("year_month").size().reset_index(name="trip_count")
print("=== 연도별 운행횟수 ===")
print(trips_ts.to_string(index=False))

In [ ]:
# 아산시 정류장 필터링 예시 (stop_name 또는 좌표 기반)
# 아산시 위경도 범위: lat 36.6~36.9 / lon 126.8~127.1 (대략)
if "stop_lat" in gtfs["stops"].columns:
    asan_stops = gtfs["stops"].query("36.6 <= stop_lat <= 36.9 and 126.8 <= stop_lon <= 127.1")
    print(f"아산시 범위 정류장: {len(asan_stops):,}개")
    print(asan_stops.groupby("year_month").size().to_string())

In [ ]:
# 컬럼 확인 (연도별 스키마 변화 확인용)
print("=== 테이블별 컬럼 ===")
for name, df in gtfs.items():
    cols = [c for c in df.columns if c not in ("year_month", "year", "month")]
    print(f"\n[{name}]")
    print("  ", cols)